# 02 · Agent → MCP server with the auth manager

Agents reach external tools and data through **MCP (Model Context Protocol)**
servers. A remote MCP server (Streamable HTTP transport) is, per the MCP
authorization spec, an **OAuth 2.0 protected resource**:

* it advertises its authorization server via
  `/.well-known/oauth-protected-resource`;
* every JSON-RPC call (`tools/list`, `tools/call`, …) must carry a valid
  **`Authorization: Bearer`** token;
* unauthenticated calls get **401** with a `WWW-Authenticate` challenge pointing
  at that metadata.

The **auth manager** supplies the token — **3-legged** when the MCP server
exposes *a user's own* data (their drive, mailbox, tickets), or **2-legged**
when it exposes org/system data behind a service identity.

> The *Runnable demo* stands up a real (mock) OAuth-protected MCP server and
> calls it through the auth manager. The *Production mapping* shows the ADK
> `MCPToolset` equivalent.


## Architecture

```
  Agent (Gemini Enterprise)                         MCP server (Streamable HTTP)
  ┌───────────────────────────┐   1. GET protected  ┌──────────────────────────┐
  │  AuthManager               │      resource meta  │ /.well-known/            │
  │   • user-drive   (3LO)     │ ───────────────────▶│   oauth-protected-resource│
  │   • org-findings (2LO)     │ ◀───────────────────│   → authorization server  │
  │  headers = auth_headers()  │   2. token by name  │                          │
  │                            │ ───────────────────▶│ POST /mcp  (JSON-RPC,     │
  └───────────────────────────┘   3. tools/call      │   Bearer …) → verify+run  │
                                                      └──────────────────────────┘
```


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from auth_provider_demo import AuthManager
from auth_provider_demo.mock_oauth_server import (
    DEMO_CLIENT_ID, DEMO_CLIENT_SECRET, MockOAuthServer,
)

RUN_PRODUCTION = False   # flip to True to run the ADK MCPToolset cell

idp = MockOAuthServer().start()

manager = AuthManager()
# 3-legged: the MCP server exposes the *user's own* documents.
manager.register_three_legged(
    "user-drive",
    token_url=idp.token_url, authorization_url=idp.authorize_url,
    client_id=DEMO_CLIENT_ID, client_secret=DEMO_CLIENT_SECRET,
    redirect_uri="https://app.example.com/oauth/callback",
    scope="mcp:read",
    description="Per-user access to the user's documents via MCP",
)
# 2-legged: an org-wide MCP server behind a service identity.
manager.register_two_legged(
    "org-findings",
    token_url=idp.token_url, client_id=DEMO_CLIENT_ID, client_secret=DEMO_CLIENT_SECRET,
    scope="mcp:read",
    description="Service identity for the org-wide findings MCP server",
)
print("authorizations:", [a["name"] for a in manager.list_authorizations()])

authorizations: ['user-drive', 'org-findings']


## Stand up a (mock) OAuth-protected MCP server

Minimal JSON-RPC MCP server: it serves protected-resource metadata, and its
`/mcp` endpoint rejects unauthenticated JSON-RPC calls, then serves `tools/list`
and `tools/call` once a valid bearer is present. The tool result reflects the
authenticated principal, so you can see 3-legged (user) vs 2-legged (service).

In [2]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def _principal_for(token: str):
    if token.startswith("user-access-"):
        return {"type": "user", "id": "alice"}
    if token.startswith("app-access-"):
        return {"type": "service", "id": "org-findings-client"}
    return None

TOOLS = [
    {"name": "list_documents",
     "description": "List documents the caller can access.",
     "inputSchema": {"type": "object", "properties": {}}},
]

def _rpc_result(id_, result): return {"jsonrpc": "2.0", "id": id_, "result": result}
def _rpc_error(id_, code, msg): return {"jsonrpc": "2.0", "id": id_,
                                        "error": {"code": code, "message": msg}}

class McpHandler(BaseHTTPRequestHandler):
    def log_message(self, *a): return

    def _send(self, code, payload, extra_headers=None):
        body = json.dumps(payload).encode()
        self.send_response(code)
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/.well-known/oauth-protected-resource":
            self._send(200, {
                "resource": MCP_BASE,
                "authorization_servers": [idp.base_url],
                "scopes_supported": ["mcp:read"],
                "bearer_methods_supported": ["header"],
            })
        else:
            self._send(404, {"error": "not_found"})

    def do_POST(self):
        if self.path != "/mcp":
            return self._send(404, {"error": "not_found"})
        auth = self.headers.get("Authorization", "")
        token = auth[7:] if auth.startswith("Bearer ") else ""
        principal = _principal_for(token)
        if principal is None:
            challenge = (f'Bearer resource_metadata='
                         f'"{MCP_BASE}/.well-known/oauth-protected-resource"')
            return self._send(401, {"error": "invalid_token"},
                              extra_headers={"WWW-Authenticate": challenge})

        length = int(self.headers.get("Content-Length", 0))
        req = json.loads(self.rfile.read(length).decode() or "{}")
        method, id_ = req.get("method"), req.get("id")
        if method == "tools/list":
            self._send(200, _rpc_result(id_, {"tools": TOOLS}))
        elif method == "tools/call":
            name = req.get("params", {}).get("name")
            if name != "list_documents":
                return self._send(200, _rpc_error(id_, -32602, f"unknown tool {name}"))
            docs = ([f"{principal['id']}/personal-notes.md", f"{principal['id']}/q3-plan.doc"]
                    if principal["type"] == "user"
                    else ["org/security-baseline.md", "org/incident-runbook.md"])
            self._send(200, _rpc_result(id_, {
                "content": [{"type": "text",
                             "text": json.dumps({"principal": principal, "documents": docs})}]
            }))
        else:
            self._send(200, _rpc_result(id_, {}))

_srv = ThreadingHTTPServer(("127.0.0.1", 0), McpHandler)
MCP_BASE = f"http://127.0.0.1:{_srv.server_address[1]}"
threading.Thread(target=_srv.serve_forever, daemon=True).start()
print("MCP server listening at", MCP_BASE)

MCP server listening at http://127.0.0.1:45523


### 1. Discover the protected-resource metadata

The client learns which authorization server issues tokens for this MCP server.

In [3]:
import urllib.request, urllib.error

def http_json(method, url, headers=None, data=None):
    req = urllib.request.Request(url, method=method, headers=headers or {})
    try:
        with urllib.request.urlopen(req, data=data) as r:
            return r.status, r.headers, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, e.headers, json.loads(e.read().decode())

_, _, meta = http_json("GET", f"{MCP_BASE}/.well-known/oauth-protected-resource")
print("authorization servers:", meta["authorization_servers"])
print("scopes supported     :", meta["scopes_supported"])

authorization servers: ['http://127.0.0.1:34023']
scopes supported     : ['mcp:read']


### 2. Unauthenticated `tools/list` is rejected

A call with no token returns **401** and a `WWW-Authenticate` challenge that
points back at the protected-resource metadata.

In [4]:
call = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}).encode()
status, hdrs, body = http_json("POST", f"{MCP_BASE}/mcp",
                               headers={"Content-Type": "application/json"}, data=call)
print("status:", status, "->", body)
print("WWW-Authenticate:", hdrs.get("WWW-Authenticate"))

status: 401 -> {'error': 'invalid_token'}
WWW-Authenticate: Bearer resource_metadata="http://127.0.0.1:45523/.well-known/oauth-protected-resource"


### 3. Call a user's MCP server with a **3-legged** token

The MCP server exposes *Alice's own* documents, so we need her authority. She
consents once; the manager then issues her token, and `tools/call` returns
**her** documents.

In [5]:
def simulate_user_consent(consent_url):
    class _NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **k): return None
    opener = urllib.request.build_opener(_NoRedirect)
    try:
        opener.open(consent_url); raise AssertionError("expected redirect")
    except urllib.error.HTTPError as e:
        return e.headers["Location"]

user = "alice"
if manager.needs_consent("user-drive", user_id=user):
    manager.complete_consent("user-drive", simulate_user_consent(
        manager.consent_url("user-drive", user_id=user)))

headers = {"Content-Type": "application/json"}
headers.update(manager.authorization_headers("user-drive", user_id=user))   # 3-legged
call = json.dumps({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                   "params": {"name": "list_documents", "arguments": {}}}).encode()
status, _, body = http_json("POST", f"{MCP_BASE}/mcp", headers=headers, data=call)
payload = json.loads(body["result"]["content"][0]["text"])
print("status:", status)
print("principal:", payload["principal"])
print("documents:", payload["documents"])

status: 200
principal: {'type': 'user', 'id': 'alice'}
documents: ['alice/personal-notes.md', 'alice/q3-plan.doc']


### 4. Call an org-wide MCP server with a **2-legged** token

For a system/org MCP server, the agent authenticates as itself; the same
`tools/call` now returns org documents under a **service** principal.

In [6]:
headers = {"Content-Type": "application/json"}
headers.update(manager.authorization_headers("org-findings"))               # 2-legged
status, _, body = http_json("POST", f"{MCP_BASE}/mcp", headers=headers, data=call)
payload = json.loads(body["result"]["content"][0]["text"])
print("status:", status)
print("principal:", payload["principal"])
print("documents:", payload["documents"])

status: 200
principal: {'type': 'service', 'id': 'org-findings-client'}
documents: ['org/security-baseline.md', 'org/incident-runbook.md']


The tool-calling code is byte-for-byte identical; only the **authorization
name** (`user-drive` vs `org-findings`) selected user vs service auth.


## Production mapping — ADK `MCPToolset`

ADK connects an agent to a remote MCP server with `MCPToolset`, passing the
connection params **and** the auth scheme/credential. The token comes from your
credential service / Gemini Enterprise authorization.

```python
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.auth import AuthCredential, AuthCredentialTypes, OAuth2Auth

mcp_tools = MCPToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://drive-mcp.internal/mcp",
    ),
    # OAuth2 scheme (authorization_code for a user-scoped MCP server) …
    auth_scheme={
        "type": "oauth2",
        "flows": {"authorizationCode": {
            "authorizationUrl": "https://accounts.google.com/o/oauth2/v2/auth",
            "tokenUrl": "https://oauth2.googleapis.com/token",
            "scopes": {"https://www.googleapis.com/auth/drive.readonly": "Read drive"},
        }},
    },
    # … and the client credential ADK uses to run it.
    auth_credential=AuthCredential(
        auth_type=AuthCredentialTypes.OAUTH2,
        oauth2=OAuth2Auth(
            client_id=os.environ["OAUTH_CLIENT_ID"],
            client_secret=os.environ["OAUTH_CLIENT_SECRET"],
        ),
    ),
)

agent = LlmAgent(
    model="gemini-2.0-flash",
    name="drive_agent",
    instruction="Answer questions about the user's documents using the MCP tools.",
    tools=[mcp_tools],
)
```

For an **org-wide (2-legged)** MCP server, swap the scheme's `authorizationCode`
flow for `clientCredentials` and reference the `org-findings` authorization.
**In Gemini Enterprise**, attach the `user-drive` / `org-findings` Authorization
from notebook 00; the platform mints and injects the MCP bearer token.


In [7]:
if RUN_PRODUCTION:
    from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
    from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
    toolset = MCPToolset(
        connection_params=StreamableHTTPConnectionParams(url=f"{MCP_BASE}/mcp"),
    )
    print("Constructed MCPToolset for", f"{MCP_BASE}/mcp")
else:
    print("RUN_PRODUCTION is False — skipping ADK MCPToolset construction.")

RUN_PRODUCTION is False — skipping ADK MCPToolset construction.


## Summary

* Remote MCP servers are **OAuth protected resources**; calls need a bearer token
  and unauthenticated calls get a **401 + `WWW-Authenticate`** pointing at
  discovery metadata.
* The **auth manager** provides the token by name — **3-legged** for a user's own
  data, **2-legged** for org/system MCP servers.
* Tool-call code stays flow-agnostic.

Next: **03_agent_gateway_psc.ipynb** — putting a private, PSC-fronted gateway in
front of these agents and MCP servers.


In [8]:
_srv.shutdown(); _srv.server_close(); idp.stop()
print("servers stopped.")

servers stopped.
